# Accessing Parsed Corpora: HeliPaD

[HeliPaD](https://github.com/DiGS-Corpora/HeliPaD) provides a fully annotated text of the Old Low German _Heliand_, but it does so in the CorpusSearch format, for which to my knowledge there are no general-purpose Python wrappers (but see [`ycoe.ipynb`](https://github.com/langeslag/ehtc/blob/main/demo/ycoe.ipynb)). Let's see if we can extract the tokens with their line references, part-of-speech tags, and lemmas. As long as we aren't interested in syntactical structures (though perhaps even then), the only library we'll need to this end is [`re`](https://docs.python.org/3/library/re.html).

In [1]:
import re,json
from pathlib import Path
from git import Repo

We'll just clone the HeliPaD repository and load its sole PSD file into a list of strings, one to a line:

In [2]:
# Loop in the HTTPS clone point here:
remote = 'https://github.com/DiGS-Corpora/HeliPaD.git'
# Desired target folder name:
local = '../corpora/HeliPaD'
# Only clone if the target folder doesn't already exist:
if not(Path(local).is_dir()):
    repo = Repo.clone_from(remote, local)
# Else, just update the working copy from remote:
else:
    repo = Repo(local)
    assert isinstance(repo, Repo)
    repo.remotes.origin.pull()
assert not repo.bare

In [3]:
with open('../corpora/HeliPaD/heliand.psd') as infile:
    psd = infile.read().splitlines()

CorpusSearch PSD files contain information on no more than one token per line. Some lines describe constituents of the syntax tree only; these we will ignore for present purposes. There are also lines with syntactical information as well as token markup; these lines use a system of nested parentheses in which the outer parentheses describe the syntactical phrase, and only the innermost pair describes the word itself, using a convention in which the part-of-speech (POS) information is followed by a space, while token form and lemma are separated by a hyphen. So this is a good opportunity to use **regular expressions**. Rather than draw up a list of the many part-of-speech tags used for annotation, we will single out sequences 

1. beginning in an opening parenthesis (we will have to escape these with a backslash), 
2. followed by a string of characters that will be our **POS tag**, 
3. followed by a space, 
4. followed by a sequence of letters that will be our **word form**, 
5. followed by a hyphen, 
6. followed by a sequence of letters that will be our **lemma**, 
7. followed by a closing parenthesis. Easy!

Regular expressions even allow us to single out **match groups** within our pattern (indicated by non-escaped parentheses), so the return contains separate containers for the three kinds of information we are after.

In [4]:

pattern = re.compile(r"\(([A-Z0-9^+=*$-]*)\s(\w*)-([^)]*)\)")

The final two pieces of information we may want to store are line number and halfline, so we can reconstruct verse lines downstream if we want.

In [5]:
line_boundary = re.compile(r"\(CODE <R_(\d*)")
caesura = re.compile(r"\(CODE <C>")

For maximum efficiency, we'll write our data to disk once we're done, and we'll loop in the option of loading it back from disk if the file is already there:

In [6]:
Path('../corpora/heliand').mkdir(parents=True, exist_ok=True)
json_file = '../corpora/heliand/heliand-c.json'
if Path(json_file).is_file():
    with open(json_file) as json_data:
        tokens = json.load(json_data)
else:
    tokens = []
    line_num = 1
    halfline = 'a'
    for line in psd:
        token = dict()
        token['verse'] = str(line_num) + halfline
        result = pattern.search(line)
        newline = line_boundary.search(line)
        off_verse = caesura.search(line)
        if result:
            token['form'] = result.group(2)
            token['lemma'] = result.group(3)
            token['pos'] = result.group(1)
            tokens.append(token)
        elif off_verse:
            halfline = 'b'
        elif newline:
            line_num = int(newline.group(1))
            halfline = 'a'
    with open(json_file, 'w', encoding='utf-8') as outfile:
        json.dump(tokens, outfile, ensure_ascii=False, indent=4)

In [7]:
tokens[205]

{'verse': '30a', 'form': 'mildean', 'lemma': 'mildi', 'pos': 'ADJ^A^SG'}

Now we can also isolate any one kind of information. For instance, we can write a lemma search function:

In [8]:
def lemsearch(lemma):
    hits = [(i['verse'], i['form']) for i in tokens if i['lemma'] == lemma]
    print(f'Hits for "{lemma}":')
    print('------------------------')
    if len(hits) < 1:
        print('No hits.')
    else:
        for hit in hits:
            print(f'{hit[0]}: {hit[1]}')
    print('------------------------')
    print(f'{len(hits)} hits total.')

In [9]:
lemsearch('kuthian')

Hits for "kuthian":
------------------------
123a: gicutdi
193a: gicuthid
399a: cuthian
432b: cuthdun
518b: cuthda
642a: gicuthdin
875b: cutda
1123b: cuthian
1285b: cuthian
1394b: cuthiat
1757b: cuthid
1797b: Kuthiat
1932b: cuthiat
2003a: gicuthda
2345b: cutda
2380b: cuthian
2426b: cuthian
3194b: cuthit
4129a: cuthdun
4657a: cuthian
5227a: cuđdi
5386b: cuthian
5403b: gicuthid
5836b: cuthian
5869a: cuthian
5920b: cuthian
5935b: cutdi
5939b: cuthian
5954a: cuthian
5963b: cuthian
------------------------
30 hits total.


Or we can write our tokens to file one (half)line at a time, reconstructing the edition ([Sievers's](https://archive.org/details/heliandherausgvonsieve)) on which the PSD file was based. Since this is the most resource-heavy routine in this notebook, we'll again check whether the file is already there, so we can skip the heavy lifting when rerunning the notebook:

In [10]:
plaintext_file = '../corpora/heliand/heliand-c.txt'
if Path(plaintext_file).is_file():
    with open(plaintext_file) as f:
        verse_lines = f.read().splitlines()
else:
    verse_lines = []
    for number in range(1, int(tokens[-1]['verse'].rstrip('ab'))):
        hits_a = [i['form'] for i in tokens if i['verse'] == str(number) + 'a']
        hits_b = [i['form'] for i in tokens if i['verse'] == str(number) + 'b']
        reconstructed_line = ' '.join(hits_a) + '    ' + ' '.join(hits_b)
        verse_lines.append(reconstructed_line)
    with open(plaintext_file, 'w') as outfile:
        outfile.write('\n'.join(verse_lines))

In [11]:
verse_lines[0]

'Manega uuaron    the sia iro mod gespon'

The edition we have just reconstructed is [Sievers 1878](https://archive.org/details/heliandherausgvonsieve), representing the C text ([London, British Library, MS Cotton Caligula A. vii](https://searcharchives.bl.uk/catalog/041-001102326)). C lacks the last fifteen lines found in the other complete witness, M ([Munich, Bayerische Staatsbibliothek, Cgm 25](https://www.digitale-sammlungen.de/de/view/bsb00026305)), and is otherwise especially notable for its many omissions of small words. For an edition of M, with C's omissions italicized and registered in the apparatus, see Behaghel's _Heliand und Genesis_, either the [current edition with Taeger](https://www.degruyterbrill.com/document/doi/10.1515/9783110963663/) or [an earlier, now public domain edition](https://archive.org/details/heliandundgenesi00beha/) for greater ease of access.

Now that we have demonstrated our method on HeliPaD, we can replicate these same techniques on other corpora in the same format: the [Icelandic Parsed Historical Corpus](http://hdl.handle.net/20.500.12537/325), for instance, or the annotated corpus of Old English verse ([YCOEP](https://www-users.york.ac.uk/~lang18/pcorpus.html)). Just keep in mind that those corpora, like YCOE but unlike HeliPaD, consist of multiple PSD files each, so you will either want to read in a single document at a time, or else load a whole folder of PSD files into a dictionary structure.

For a demonstration of how the data in a corpus like HeliPaD may be put to use, now see [Helix: A _Heliand_ Concordance](https://langeslag.uni-goettingen.de/helix/).